## Definitions

Let subscripts denote nodal or stencil and superscripts denote edge quantites. For an edge $e^i$, let

- $\mathbf{t}^i$ be the tangent vector.
- $\{\mathbf{d}_1^i, \mathbf{d}_2^i, \mathbf{t}^i\}$ be the orthonormal reference frame.
- $\{\mathbf{m}_1^i, \mathbf{m}_2^i, \mathbf{t}^i\}$ be the orthonormal material frame and the rotation of reference frame by a signed angle $\theta^i$ along $\mathbf{t}^i$.

## Formulation

For a 11 node stencil, let

- $\mathbf{q}=[x_0, y_0, z_0, \theta^0, x_1, y_1, z_1, \theta^1, x_2, y_2, z_2]^T$ be the state vector.
- $\mathbf{aux}=[\mathbf{t}^0_{\textrm{old}}, \mathbf{t}^1_{\textrm{old}}, \mathbf{d}_{1,\textrm{old}}^0, \mathbf{d}_{1,\textrm{old}}^1, \beta_\textrm{old}]^T$ be the auxiliary vector.
- $\boldsymbol{\epsilon}=[\epsilon^0, \epsilon^1, \kappa_1, \kappa_2, \tau]^T$ be the strain vector.
- $E=f(\boldsymbol{\epsilon})$ be the energy.

### Frame Reconstruction

First, we calcualte the $\mathbf{d}_{1,\textrm{new}}^i$ from $\mathbf{aux}$

$$
\mathbf{d}_{1,\textrm{new}}^i=\textrm{ParallelTransport}(\mathbf{d}_{1,\textrm{old}}^i,\mathbf{t}_{\textrm{old}}^i, \mathbf{t}_{\textrm{new}}^i)
$$

With $\mathbf{t}_{\textrm{new}}^i$ and $\mathbf{d}_{1,\textrm{new}}^i$ we construct the **new** orthonormal reference frame $\{\mathbf{d}_1^i, \mathbf{d}_2^i, \mathbf{t}^i\}$ as

$$
\mathbf{d}_{2,\textrm{new}}^i=\mathbf{t}^i_{\textrm{new}}\times \mathbf{d}_{1,\textrm{new}}^i
$$

With $\{\mathbf{d}_1^i, \mathbf{d}_2^i, \mathbf{t}^i\}$ and $\theta^i$, we compute the material frame $\{\mathbf{m}_1^i, \mathbf{m}_2^i, \mathbf{t}^i\}$

$$
\begin{align*}
m_1^i&=\cos(\theta^i)\mathbf{d}_1^i+\sin(\theta^i)\mathbf{d}_2^i \\
m_2^i&=-\sin(\theta^i)\mathbf{d}_1^i+\cos(\theta^i)\mathbf{d}_2^i
\end{align*}
$$

### Strains

#### Stretching Strain 

The scalar stretching strain is the ratio of elongation / compression of an edge:

$$
\epsilon^i=\frac{\|\mathbf{e}^i\|}{\|\mathbf{\bar e}^i\|}-1
$$

#### Bending Strains

The discrete curvature binormal $(\kappa\mathbf{b})$ at node $i$ represents the integrated curvature between edges $\mathbf{e}^0$ and $\mathbf{e}^1$:

$$
(\kappa\mathbf{b})=\frac{2(\mathbf{t^0}\times \mathbf{t^1})}{1+\mathbf{t^0}\cdot \mathbf{t^1}}
$$

The scalar bending strains are the projections of this binormal onto the averaged material directors:

$$
\begin{align*}
\kappa_1&=\frac{1}{2}(\kappa\mathbf{b})\cdot(\mathbf{m}_2^0+\mathbf{m}_2^1) \\
\kappa_2&=-\frac{1}{2}(\kappa\mathbf{b})\cdot(\mathbf{m}_1^0+\mathbf{m}_1^1) 
\end{align*}
$$

#### Twisting Strain

The twisting strain $\tau$ accounts for the relative rotation of material frames plus the geometric holonomy:

$$
\tau=\theta^1-\theta^0+\beta
$$

Where $\beta$ is the angle of parallel transport between reference frames. For $\mathbf{t}^0 \cdot \mathbf{t}^1 > -1$:

$$\beta = \operatorname{atan2}\left( (\mathbf{d}_{1,\textrm{new}}^0 \times \mathbf{d}_{1,\textrm{new}}^1) \cdot \mathbf{\bar{t}}, \,\, \mathbf{d}_{1,\textrm{new}}^0 \cdot \mathbf{d}_{1,\textrm{new}}^1 \right), \quad \mathbf{\bar{t}} = \frac{\mathbf{t}^0 \times \mathbf{t}^1}{\|\mathbf{t}^0 \times \mathbf{t}^1\|}$$

### Energy

$$
E=f(\boldsymbol{\epsilon})
$$


In [13]:
@wp.kernel
def kappa_der(
    nodes: wp.array(dtype=wp.vec3),
    m1s: wp.array(dtype=wp.vec3),
    m2s: wp.array(dtype=wp.vec3),
    ks: wp.array(dtype=float),
    E: wp.array(dtype=float),
) -> wp.vec2:
    idx = wp.tid()
    n0, n1, n2 = nodes[idx], nodes[idx + 1], nodes[idx + 2]
    m1e, m2e = m1s[idx], m2s[idx]
    m1f, m2f = m1s[idx + 1], m2s[idx + 1]
    k1, k2 = ks[idx], ks[idx + 1]

    ee = n1 - n0
    ef = n2 - n1
    te = wp.normalize(ee)
    tf = wp.normalize(ef)
    inv_denom = 1.0 / (1.0 + wp.dot(te, tf))  # TODO: add eps?
    kb = 2.0 * wp.cross(te, tf) * inv_denom
    kappa1 = 0.5 * wp.dot(kb, m2e + m2f)
    kappa2 = -0.5 * wp.dot(kb, m1e + m1f)
    E[idx] = 0.5 * (k1 * kappa1**2 + k2 * kappa2**2)


@wp.kernel
def grad_kappa_der(
    nodes: wp.array(dtype=wp.vec3),
    m1s: wp.array(dtype=wp.vec3),
    m2s: wp.array(dtype=wp.vec3),
    ks: wp.array(dtype=float),
    F: wp.array(dtype=float),
):
    idx = wp.tid()
    n0, n1, n2 = nodes[idx], nodes[idx + 1], nodes[idx + 2]
    m1e, m2e = m1s[idx], m2s[idx]
    m1f, m2f = m1s[idx + 1], m2s[idx + 1]
    k1, k2 = ks[idx], ks[idx + 1]

    ee = n1 - n0
    ef = n2 - n1
    inv_ee = 1.0 / wp.length(ee)
    inv_ef = 1.0 / wp.length(ef)
    te = ee * inv_ee
    tf = ef * inv_ef

    inv_denom = 1.0 / (1.0 + wp.dot(te, tf))  # TODO: add eps?
    kb = 2.0 * wp.cross(te, tf) * inv_denom

    kappa1 = 0.5 * wp.dot(kb, m2e + m2f)
    kappa2 = -0.5 * wp.dot(kb, m1e + m1f)

    k1_factor = k1 * kappa1
    k2_factor = k2 * kappa2
    tilde_t = (te + tf) * inv_denom
    tilde_d1 = (m1e + m1f) * inv_denom
    tidle_d2 = (m2e + m2f) * inv_denom
    de = k1_factor * inv_ee * (-kappa1 * tilde_t + wp.cross(tf, tidle_d2))
    de += k2_factor * inv_ee * (-kappa2 * tilde_t - wp.cross(tf, tilde_d1))
    df = k1_factor * inv_ef * (-kappa1 * tilde_t + wp.cross(tf, tidle_d2))
    df += k2_factor * inv_ef * (-kappa2 * tilde_t - wp.cross(tf, tilde_d1))
    dthetae = k1_factor * -0.5 * wp.dot(kb, m1e)
    dthetae += k2_factor * -0.5 * wp.dot(kb, m2e)
    dthetaf = k1_factor * -0.5 * wp.dot(kb, m1f)
    dthetaf += k2_factor * -0.5 * wp.dot(kb, m2f)

    base = idx * 4  # skip a node

    wp.atomic_add(F, base + 0, -de[0])
    wp.atomic_add(F, base + 1, -de[1])
    wp.atomic_add(F, base + 2, -de[2])
    wp.atomic_add(F, base + 3, dthetae)
    wp.atomic_add(F, base + 4, de[0] - df[0])
    wp.atomic_add(F, base + 5, de[1] - df[1])
    wp.atomic_add(F, base + 6, de[2] - df[2])
    wp.atomic_add(F, base + 7, dthetaf)
    wp.atomic_add(F, base + 8, df[0])
    wp.atomic_add(F, base + 9, df[1])
    wp.atomic_add(F, base + 10, df[2])


In [22]:
import warp as wp
import numpy as np

wp.init()


# --- Your Kernels ---
@wp.kernel
def kappa_der(
    nodes: wp.array(dtype=wp.vec3),
    m1s: wp.array(dtype=wp.vec3),
    m2s: wp.array(dtype=wp.vec3),
    ks: wp.array(dtype=float),
    E: wp.array(dtype=float),
) -> wp.vec2:
    idx = wp.tid()
    n0, n1, n2 = nodes[idx], nodes[idx + 1], nodes[idx + 2]
    m1e, m2e = m1s[idx], m2s[idx]
    m1f, m2f = m1s[idx + 1], m2s[idx + 1]
    k1, k2 = ks[idx], ks[idx + 1]

    ee = n1 - n0
    ef = n2 - n1
    te = wp.normalize(ee)
    tf = wp.normalize(ef)
    inv_denom = 1.0 / (1.0 + wp.dot(te, tf))
    kb = 2.0 * wp.cross(te, tf) * inv_denom
    kappa1 = 0.5 * wp.dot(kb, m2e + m2f)
    kappa2 = -0.5 * wp.dot(kb, m1e + m1f)

    # Store total energy for this stencil
    E[idx] = 0.5 * (k1 * kappa1 * kappa1 + k2 * kappa2 * kappa2)


@wp.kernel
def grad_kappa_der(
    nodes: wp.array(dtype=wp.vec3),
    m1s: wp.array(dtype=wp.vec3),
    m2s: wp.array(dtype=wp.vec3),
    ks: wp.array(dtype=float),
    F: wp.array(dtype=float),
):
    idx = wp.tid()
    n0, n1, n2 = nodes[idx], nodes[idx + 1], nodes[idx + 2]
    m1e, m2e = m1s[idx], m2s[idx]
    m1f, m2f = m1s[idx + 1], m2s[idx + 1]
    k1, k2 = ks[idx], ks[idx + 1]

    ee = n1 - n0
    ef = n2 - n1
    inv_ee = 1.0 / wp.length(ee)
    inv_ef = 1.0 / wp.length(ef)
    te = ee * inv_ee
    tf = ef * inv_ef

    inv_denom = 1.0 / (1.0 + wp.dot(te, tf))
    kb = 2.0 * wp.cross(te, tf) * inv_denom

    kappa1 = 0.5 * wp.dot(kb, m2e + m2f)
    kappa2 = -0.5 * wp.dot(kb, m1e + m1f)

    k1_factor = k1 * kappa1
    k2_factor = k2 * kappa2
    tilde_t = (te + tf) * inv_denom
    tilde_d1 = (m1e + m1f) * inv_denom
    tidle_d2 = (m2e + m2f) * inv_denom
    de = k1_factor * inv_ee * (-kappa1 * tilde_t + wp.cross(tf, tidle_d2))
    de += k2_factor * inv_ee * (-kappa2 * tilde_t - wp.cross(tf, tilde_d1))
    df = k1_factor * inv_ef * (-kappa1 * tilde_t - wp.cross(te, tidle_d2))
    df += k2_factor * inv_ef * (-kappa2 * tilde_t + wp.cross(te, tilde_d1))
    dthetae = k1_factor * -0.5 * wp.dot(kb, m1e)
    dthetae += k2_factor * -0.5 * wp.dot(kb, m2e)
    dthetaf = k1_factor * -0.5 * wp.dot(kb, m1f)
    dthetaf += k2_factor * -0.5 * wp.dot(kb, m2f)

    base = idx * 4

    wp.atomic_add(F, base + 0, -de[0])
    wp.atomic_add(F, base + 1, -de[1])
    wp.atomic_add(F, base + 2, -de[2])
    wp.atomic_add(F, base + 3, dthetae)
    wp.atomic_add(F, base + 4, de[0] - df[0])
    wp.atomic_add(F, base + 5, de[1] - df[1])
    wp.atomic_add(F, base + 6, de[2] - df[2])
    wp.atomic_add(F, base + 7, dthetaf)
    wp.atomic_add(F, base + 8, df[0])
    wp.atomic_add(F, base + 9, df[1])
    wp.atomic_add(F, base + 10, df[2])


def normalize(v):
    norm = np.linalg.norm(v)
    if norm < 1e-10:
        return v
    return v / norm


def parallel_transport(u, t0, t1):
    """
    Transports vector u (which is orthogonal to t0) to be orthogonal to t1
    by rotating it along the 'shortest path' geodesic.
    """
    b = np.cross(t0, t1)
    if np.linalg.norm(b) < 1e-10:
        return u
    b = normalize(b)
    n0 = np.linalg.norm(t0)
    n1 = np.linalg.norm(t1)
    # Angle between t0 and t1
    cos_phi = np.dot(t0, t1)
    # Clip for safety
    cos_phi = np.clip(cos_phi, -1.0, 1.0)
    phi = np.arccos(cos_phi)

    # Rodrigues rotation formula
    # v_rot = v cos(a) + (k x v) sin(a) + k (k . v) (1 - cos(a))
    # Here k is b.
    return (
        u * np.cos(phi)
        + np.cross(b, u) * np.sin(phi)
        + b * np.dot(b, u) * (1 - np.cos(phi))
    )


def rotate_vector(v, axis, angle):
    """Rotate vector v around axis by angle."""
    axis = normalize(axis)
    return (
        v * np.cos(angle)
        + np.cross(axis, v) * np.sin(angle)
        + axis * np.dot(axis, v) * (1 - np.cos(angle))
    )


def run_fdm_check():
    print("--- Discrete Elastic Rod Gradient Check ---")
    device = "cuda" if wp.is_cuda_available() else "cpu"

    # Setup 3 nodes (1 internal hinge)
    # Bent configuration to ensure gradients are non-zero
    nodes_np = np.array(
        [[0.0, 0.0, 0.0], [1.0, 0.0, 0.0], [1.8, 0.5, 0.2]], dtype=np.float32
    )

    num_nodes = 3
    num_edges = 2

    # Setup Tangents
    e0 = nodes_np[1] - nodes_np[0]
    e1 = nodes_np[2] - nodes_np[1]
    t0 = normalize(e0)
    t1 = normalize(e1)

    # Initialize Directors (Orthonormal to tangent)
    # Arbitrary reference
    up = np.array([0.0, 1.0, 0.0], dtype=np.float32)

    m1_np = np.zeros((num_edges, 3), dtype=np.float32)
    m2_np = np.zeros((num_edges, 3), dtype=np.float32)

    # Edge 0
    m1_np[0] = normalize(np.cross(t0, up))
    m2_np[0] = normalize(np.cross(t0, m1_np[0]))

    # Edge 1
    m1_np[1] = normalize(np.cross(t1, up))
    m2_np[1] = normalize(np.cross(t1, m1_np[1]))

    # Stiffness params
    ks_np = np.array([10.0, 20.0], dtype=np.float32)

    # Warp Arrays
    nodes = wp.array(nodes_np, dtype=wp.vec3, device=device)
    m1s = wp.array(m1_np, dtype=wp.vec3, device=device)
    m2s = wp.array(m2_np, dtype=wp.vec3, device=device)
    ks = wp.array(ks_np, dtype=float, device=device)

    # Output Buffers
    E_buf = wp.zeros(1, dtype=float, device=device)  # We only have 1 hinge
    F_analytic = wp.zeros(num_nodes * 4, dtype=float, device=device)

    # 1. Analytic Force
    wp.launch(
        grad_kappa_der, dim=1, inputs=[nodes, m1s, m2s, ks, F_analytic], device=device
    )
    F_warp = F_analytic.numpy()

    # 2. Finite Difference Check
    eps = 1e-3
    F_fdm = np.zeros_like(F_warp)

    # We will perturb every DOF in the system: [x0, y0, z0, theta0, x1, y1, z1, theta1, ...]
    # DOFs 3, 7, 11 are Thetas. Rest are Positions.

    print(f"Running FDM with eps={eps}...")

    for i in range(num_nodes * 4):
        node_idx = i // 4
        dof_type = i % 4  # 0,1,2=Pos, 3=Theta

        # --- Prepare Perturbed State ---
        nodes_plus = nodes_np.copy()
        nodes_minus = nodes_np.copy()
        m1_plus = m1_np.copy()
        m2_plus = m2_np.copy()
        m1_minus = m1_np.copy()
        m2_minus = m2_np.copy()

        if dof_type == 3:
            # === Theta Perturbation ===
            # Rotate m1, m2 around their respective tangent
            edge_idx = min(
                node_idx, num_edges - 1
            )  # theta is usually associated with the edge
            # Logic: In the layout, idx=3 is theta_0 (edge 0), idx=7 is theta_1 (edge 1)
            # But the kernel writes to idx*4 + 3 and (idx+1)*4 + 7
            # So dof 3 is indeed theta_0, dof 7 is theta_1.

            # Plus
            t = normalize(nodes_np[edge_idx + 1] - nodes_np[edge_idx])
            m1_plus[edge_idx] = rotate_vector(m1_np[edge_idx], t, eps)
            m2_plus[edge_idx] = rotate_vector(m2_np[edge_idx], t, eps)

            # Minus
            m1_minus[edge_idx] = rotate_vector(m1_np[edge_idx], t, -eps)
            m2_minus[edge_idx] = rotate_vector(m2_np[edge_idx], t, -eps)

        else:
            # === Node Perturbation ===
            axis = dof_type

            # Move Node +
            nodes_plus[node_idx, axis] += eps

            # Move Node -
            nodes_minus[node_idx, axis] -= eps

            # **CRITICAL**: Update Frames via Parallel Transport
            # Since nodes moved, tangents changed. We must transport m1, m2 to new tangents.
            # We do this for ALL edges affected by this node.

            for edges, n_arr, m1_arr, m2_arr in [
                (nodes_plus, nodes_plus, m1_plus, m2_plus),
                (nodes_minus, nodes_minus, m1_minus, m2_minus),
            ]:
                # Recompute tangents and transport
                for e_i in range(num_edges):
                    t_orig = normalize(nodes_np[e_i + 1] - nodes_np[e_i])
                    t_new = normalize(n_arr[e_i + 1] - n_arr[e_i])

                    m1_arr[e_i] = parallel_transport(m1_np[e_i], t_orig, t_new)
                    m2_arr[e_i] = parallel_transport(m2_np[e_i], t_orig, t_new)

        # --- Compute Energy Plus ---
        wp.copy(nodes, wp.array(nodes_plus, dtype=wp.vec3, device=device))
        wp.copy(m1s, wp.array(m1_plus, dtype=wp.vec3, device=device))
        wp.copy(m2s, wp.array(m2_plus, dtype=wp.vec3, device=device))

        E_buf.zero_()
        wp.launch(kappa_der, dim=1, inputs=[nodes, m1s, m2s, ks, E_buf], device=device)
        e_plus = E_buf.numpy()[0]

        # --- Compute Energy Minus ---
        wp.copy(nodes, wp.array(nodes_minus, dtype=wp.vec3, device=device))
        wp.copy(m1s, wp.array(m1_minus, dtype=wp.vec3, device=device))
        wp.copy(m2s, wp.array(m2_minus, dtype=wp.vec3, device=device))

        E_buf.zero_()
        wp.launch(kappa_der, dim=1, inputs=[nodes, m1s, m2s, ks, E_buf], device=device)
        e_minus = E_buf.numpy()[0]

        # FDM Gradient = (E+ - E-) / 2eps
        # Force = - Gradient
        F_fdm[i] = (e_plus - e_minus) / (2.0 * eps)

    # Compare
    # Note: Indices 11 (theta 2) is not written to by the kernel (only 0..10 are touched for 1 hinge)
    # So we only check first 11 indices.

    valid_mask = np.arange(num_nodes * 4) < 11

    diff = np.abs(F_warp[valid_mask] - F_fdm[valid_mask])
    max_diff = np.max(diff)

    print("\nComparison (Analytic Force vs -FDM Grad):")
    # Print a few for debugging
    for i in range(11):
        type_str = "Theta" if i % 4 == 3 else "Pos  "
        print(
            f"DOF {i:02d} [{type_str}]: Analytic={F_warp[i]:.5f}, FDM={F_fdm[i]:.5f}, Diff={diff[i]:.5e}"
        )

    print(f"\nMax Diff: {max_diff:.6e}")

    if max_diff < 1e-2:  # Relaxed tolerance for float32 + complex stencil
        print(">> VERIFIED!")
    else:
        print(">> MISMATCH")


if __name__ == "__main__":
    run_fdm_check()

--- Discrete Elastic Rod Gradient Check ---
Module __main__ 21c0d5d load on device 'cuda:0' took 1118.24 ms  (compiled)
Running FDM with eps=0.001...

Comparison (Analytic Force vs -FDM Grad):
DOF 00 [Pos  ]: Analytic=0.00000, FDM=0.00000, Diff=6.28918e-07
DOF 01 [Pos  ]: Analytic=12.22000, FDM=12.22014, Diff=1.43051e-04
DOF 02 [Pos  ]: Analytic=2.42684, FDM=2.42662, Diff=2.10762e-04
DOF 03 [Theta]: Analytic=0.56885, FDM=0.56899, Diff=1.36018e-04
DOF 04 [Pos  ]: Analytic=7.09179, FDM=7.09248, Diff=6.82831e-04
DOF 05 [Pos  ]: Analytic=-22.85416, FDM=-22.85337, Diff=7.87735e-04
DOF 06 [Pos  ]: Analytic=-4.20860, FDM=-4.20868, Diff=8.20160e-05
DOF 07 [Theta]: Analytic=0.80416, FDM=0.80419, Diff=2.70009e-05
DOF 08 [Pos  ]: Analytic=-7.09179, FDM=-7.09271, Diff=9.20773e-04
DOF 09 [Pos  ]: Analytic=10.63416, FDM=10.63383, Diff=3.35693e-04
DOF 10 [Pos  ]: Analytic=1.78177, FDM=1.78182, Diff=5.43594e-05

Max Diff: 9.207726e-04
>> VERIFIED!


In [ ]:
@wp.kernel
def hess_kappa_der(
    nodes: wp.array(dtype=wp.vec3),
    m1s: wp.array(dtype=wp.vec3),
    m2s: wp.array(dtype=wp.vec3),
    ks: wp.array(dtype=float),
    F: wp.array(dtype=float),
):
    idx = wp.tid()
    n0, n1, n2 = nodes[idx], nodes[idx + 1], nodes[idx + 2]
    m1e, m2e = m1s[idx * 2], m2s[idx * 2]
    m1f, m2f = m1s[idx * 2 + 1], m2s[idx * 2 + 1]

    ee = n1 - n0
    ef = n2 - n1
    inv_ee = 1.0 / wp.length(ee)
    inv_ef = 1.0 / wp.length(ef)
    te = ee * inv_ee
    tf = ef * inv_ef

    inv_denom = 1.0 / (1.0 + wp.dot(te, tf))  # TODO: add eps?
    kb = 2.0 * wp.cross(te, tf) * inv_denom

    kappa1 = 0.5 * wp.dot(kb, m2e + m2f)
    kappa2 = -0.5 * wp.dot(kb, m1e + m1f)

    tilde_t = (te + tf) * inv_denom
    # tilde_d1 = (m1e + m1f) * inv_denom
    tilde_d2 = (m2e + m2f) * inv_denom

    tt_o_tt = wp.outer(tilde_t, tilde_t)
    tf_c_d2t_o_tt = wp.outer(wp.cross(tf, tilde_d2), tilde_t)
    tt_o_tf_c_d2t = wp.transpose(tf_c_d2t_o_tt)
    kb_o_d2e = wp.outer(kb, m2e)
    d2e_o_kb = wp.transpose(kb_o_d2e)

    I3 = wp.mat33(1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0)

    D2kappa1De2 = (
        inv_ee * inv_ee * (2.0 * kappa1 * tt_o_tt - tf_c_d2t_o_tt - tt_o_tf_c_d2t)
        - kappa1 * inv_denom * inv_ee * inv_ee * (I3 - wp.outer(te, te))
        + 0.25 * inv_ee * inv_ee * (kb_o_d2e + d2e_o_kb)
    )

    te_c_d2t_o_tt = wp.outer(wp.cross(te, tilde_d2), tilde_t)
    tt_o_te_c_d2t = wp.transpose(te_c_d2t_o_tt)
    kb_o_d2f = wp.outer(kb, m2f)
    d2f_o_kb = wp.transpose(kb_o_d2f)
    tf_o_tf = wp.outer(tf, tf)

    D2kappa1Df2 = (
        inv_ef * inv_ef * (2.0 * kappa1 * tt_o_tt + te_c_d2t_o_tt + tt_o_te_c_d2t)
        - kappa1 * inv_denom * inv_ee * inv_ee * (I3 - tf_o_tf)
        + 0.25 * inv_ef * inv_ef * (kb_o_d2f + d2f_o_kb)
    )

    D2kappa1DeDf = -kappa1 * inv_denom * inv_ee * inv_ef * (I3 + wp.outer(te, tf))
    +1.0 * inv_ee * inv_ef * (
        2.0 * kappa1 * tt_o_tt - tf_c_d2t_o_tt + tt_o_te_c_d2t
        #    - crossMat(tilde_d2)  # todo cross mat
    )
    D2kappa1DfDe = wp.transpose(D2kappa1DeDf)

    D2kappa1Dthetae = -0.5 * wp.dot(kb, m2e)
    D2kappa1Dthetaf2 = -0.5 * wp.dot(kb, m2f)
    D2kappa2Dthetae2 = 0.5 * wp.dot(kb, m1e)
    D2kappa2Dthetaf2 = 0.5 * wp.dot(kb, m1f)

    D2kappa1DeDthetae = inv_ee * (
        0.5 * wp.dot(kb, m1e) * tilde_t - inv_denom * wp.cross(tf, m1e)
    )
    D2kappa1DeDthetaf = inv_ee * (
        0.5 * wp.dot(kb, m1f) * tilde_t - inv_denom * wp.cross(tf, m1f)
    )
    D2kappa1DfDthetae = inv_ef * (
        0.5 * wp.dot(kb, m1e) * tilde_t + inv_denom * wp.cross(te, m1e)
    )
    D2kappa1DfDthetaf = inv_ef * (
        0.5 * wp.dot(kb, m1f) * tilde_t + inv_denom * wp.cross(te, m1f)
    )
    D2kappa2DeDthetae = inv_ee * (
        0.5 * wp.dot(kb, m2e) * tilde_t - inv_denom * wp.cross(tf, m2e)
    )
    D2kappa2DeDthetaf = inv_ee * (
        0.5 * wp.dot(kb, m2f) * tilde_t - inv_denom * wp.cross(tf, m2f)
    )
    D2kappa2DfDthetae = inv_ef * (
        0.5 * wp.dot(kb, m2e) * tilde_t + inv_denom * wp.cross(te, m2e)
    )
    D2kappa2DfDthetaf = inv_ef * (
        0.5 * wp.dot(kb, m2f) * tilde_t + inv_denom * wp.cross(te, m2f)
    )
